# Lab 3: Recognising Activities from Phone Sensors (RNN vs LSTM vs GRU)

**Hands-on lab · Recurrent neural networks on real sensor sequences**

Your phone knows when you're walking. Fitness apps count steps and active minutes, health apps track how long older users sit still, and fall-detection features alert family when someone suddenly goes from upright to lying down. All of these start from the same raw material: the phone's **accelerometer** (which measures acceleration, including the constant pull of gravity) and **gyroscope** (which measures how fast the phone is rotating), which report how it moves and turns, dozens of times a second.

Unlike a table row or a photo, this data is a **sequence**: what matters is how the readings change over time. A step is a rhythm, and climbing stairs has a different rhythm from walking on the flat. That makes it a natural task for **recurrent neural networks**, which read a sequence one step at a time and carry a memory forward.

In this lab we build an activity recogniser from real phone recordings and use it to compare the three classic recurrent networks: the plain **RNN**, the **LSTM** and the **GRU**. We compare them on accuracy, size and training time, and on the question that matters most in practice: does the model work for *people it has never seen*?

**What you'll see in this lab:**
- real smartphone sensor recordings from 30 people, cut into 2.56-second windows;
- a split by *person*, so the test measures performance on new people;
- a simple statistics baseline, then RNN, LSTM and GRU models trained on the raw sequences;
- a fair comparison: accuracy, parameters, training time, and variation across random seeds;
- how accuracy varies from person to person, and which activities get confused;
- a working app that names the activity in a recording.

**The data.** The *Human Activity Recognition Using Smartphones* dataset (UCI Machine Learning Repository, CC BY 4.0). Thirty volunteers carried a smartphone at the waist while walking, walking upstairs, walking downstairs, sitting, standing and lying down. The phone recorded acceleration and rotation 50 times a second, and the recordings were cut into windows of 128 readings (2.56 seconds). The original authors split the volunteers into a training group and a separate test group of different people. We load a copy of the dataset from the Hugging Face Hub, which is much faster to download than the original site.

Two words we use throughout: a **window** is one 2.56-second slice of recording (one example for the model), and a **channel** is one of the 9 sensor signals recorded at every time step (for example, acceleration along the x axis).

### Running this lab

- The **first run** trains all the models and saves them to `artifacts/` (a few minutes on a laptop CPU, including a check with extra random seeds). **Later runs load the saved models**. To retrain, delete `artifacts/` or set `TRAIN = True`.
- **`LAUNCH_APP`** in the last cell opens the app here; `python app.py` runs it on its own.

### How to use this notebook

- **Run the cells in order, from top to bottom** (in Jupyter: *Run All*, or Shift+Enter cell by cell). Later cells use variables created by earlier ones, so skipping a cell causes errors.
- **The `TRAIN` flag** is set automatically in the first code cell: it is `True` only when no saved model exists yet. On the first run the notebook downloads the data, trains the models and saves everything to `artifacts/`; on every later run it loads those files instead, so the whole notebook takes seconds. Because of this, the learning-curve plot in Section 4 only appears on a run that actually trains.
- **The code is commented line by line.** Read the comments together with the text: the text says *what* we are doing and *why*; the comments say *how*.
- **Try it yourself** boxes are optional experiments. Several of them retrain models, which overwrites the files in `artifacts/`; copy that folder somewhere safe first if you want to keep the original results.

### Roadmap

| Section | What happens | What it produces |
|---|---|---|
| Setup (next cell) | Imports, random seeds, the `TRAIN` flag | The device and whether we train or load |
| 1. The Data | Download and load the raw windows; plot one per activity | Arrays of shape (windows, 128, 9), labels and person ids |
| 2. Splitting by Person | Hold out 4 whole people for validation; scale the channels | Training and validation sets, scaling statistics |
| 3. A Baseline Without Sequences | Summary statistics + logistic regression | The accuracy to beat |
| 4. Three Recurrent Networks | Define, train (or load) RNN, LSTM and GRU; choose one on validation | Saved models, the chosen model, learning curves |
| 5. Comparing on New People | Test accuracy per activity, confusion matrix, seed study | The fair comparison |
| 6. Does It Work for Everyone? | Accuracy for each test person | The spread across people |
| 7. The Application | A prediction function and a Gradio app | Files the app needs, example predictions |
| What We Learned | Recap, results table and key terms | |

The first code cell loads the libraries, fixes the random **seed** (the starting value of the random number generator, so random choices come out the same on every run) and decides whether to train or load.

In [21]:
# --- Setup: libraries, the TRAIN switch, random seeds and the device ---
import os, json, time                       # files and folders, saving small results, timing
import numpy as np                          # arrays for the raw sensor data
import torch                                # PyTorch: tensors and automatic gradients
import torch.nn as nn                       # neural-network layers (RNN, LSTM, GRU, Linear)
import torch.nn.functional as F             # loss functions such as cross-entropy
import matplotlib.pyplot as plt             # plots
from sklearn.linear_model import LogisticRegression    # the simple, non-sequence baseline
from huggingface_hub import snapshot_download          # downloads the dataset copy from the Hugging Face Hub

# TRAIN is True only if no saved GRU model exists yet. So the first run trains everything (a few minutes),
# and every later run just loads the saved models from artifacts/ (seconds). Set TRAIN = True to force retraining.
TRAIN = not os.path.exists("artifacts/gru.pt")              # train the first time; afterwards load the saved models
# A "seed" fixes the random number generator, so random weights and shuffles are the same on every run.
# That makes results reproducible: you should see the same numbers as the instructor.
torch.manual_seed(0)
np.random.seed(0)
# Use a GPU if one is available; otherwise the CPU (these small models train fine on a CPU).
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device, "| train now:", TRAIN)

PyTorch 2.14.1 | device: cpu | train now: False


**How to read this output:** it shows your PyTorch version, whether you are using a GPU (`cuda`) or the CPU, and `train now: True` (first run, will train) or `False` (will load saved models). A warning about `IProgress` or `tqdm` is harmless and can be ignored.

## 1. The Data

The next cell downloads the dataset on the first run only, then reads the raw sensor files into one array per split. Each array has the shape *(number of windows, 128 time steps, 9 channels)*, and comes with the activity label and the person id of every window.

In [22]:
# --- Load the UCI HAR raw sensor windows (download once, then read from disk) ---
ROOT = "data/UCI_HAR"
# Only download if the files are not already here, so later runs work offline and start quickly.
if not os.path.exists(f"{ROOT}/train/y_train.txt"):         # a copy of the UCI dataset on the Hugging Face Hub
    # allow_patterns downloads only what we need: the raw "Inertial Signals", the labels, the person ids
    # and the activity names. We skip the 561 hand-made features, because our networks read the raw sequence.
    snapshot_download("udayl/UCI_HAR", repo_type="dataset", local_dir=ROOT,
                      allow_patterns=["*/Inertial Signals/*", "*/y_*.txt", "*/subject_*.txt", "activity_labels.txt"])
# The 9 channels, in a fixed order: body acceleration (gravity removed), gyroscope rotation,
# and total acceleration (including gravity), each along x, y and z.
SIGNALS = ["body_acc_x", "body_acc_y", "body_acc_z", "body_gyro_x", "body_gyro_y", "body_gyro_z",
           "total_acc_x", "total_acc_y", "total_acc_z"]
# Activity names, e.g. "WALKING_UPSTAIRS" -> "walking upstairs". Label k in the data is ACTIVITIES[k].
ACTIVITIES = [line.split()[1].lower().replace("_", " ") for line in open(f"{ROOT}/activity_labels.txt")]

def load(split):
    """Load one split ("train" or "test") of the raw sensor windows.

    Returns:
        X:    float32 array (N, 128, 9) - N windows, 128 time steps (2.56 s at 50 Hz), 9 sensor channels.
        y:    int array (N,) - activity label 0-5 (the files store 1-6, so we subtract 1).
        subj: int array (N,) - which volunteer (person id 1-30) recorded each window.
    Each channel lives in its own text file of shape (N, 128); stacking them on the last axis gives (N, 128, 9).
    """
    X = np.stack([np.loadtxt(f"{ROOT}/{split}/Inertial Signals/{s}_{split}.txt") for s in SIGNALS], axis=-1)  # (N, 128, 9)
    y = np.loadtxt(f"{ROOT}/{split}/y_{split}.txt").astype(int) - 1     # labels 1-6 -> 0-5 (PyTorch counts from 0)
    subj = np.loadtxt(f"{ROOT}/{split}/subject_{split}.txt").astype(int)  # person id: needed to split by person
    return X.astype(np.float32), y, subj                    # float32 is what PyTorch models expect

t0 = time.perf_counter()
X_all, y_all, s_all = load("train")             # the original training group of people
X_test, y_test, s_test = load("test")           # the original test group: completely different people
print(f"training windows {X_all.shape} | test windows {X_test.shape} | loaded in {time.perf_counter() - t0:.0f} s")
print(f"people: {len(set(s_all))} in training, {len(set(s_test))} different people in test")
# Count the windows per activity, to check the classes are roughly balanced.
print("activities:", {a: int((y_all == k).sum()) for k, a in enumerate(ACTIVITIES)})

training windows (7352, 128, 9) | test windows (2947, 128, 9) | loaded in 1 s
people: 21 in training, 9 different people in test
activities: {'walking': 1226, 'walking upstairs': 1073, 'walking downstairs': 986, 'sitting': 1286, 'standing': 1374, 'laying': 1407}


There are 7352 training windows and 2947 test windows. Each window is 128 time steps × 9 channels: body acceleration and rotation along three axes, plus total acceleration, which includes gravity, along three axes. The 21 training people and the 9 test people are different people. The six activities are roughly balanced.

**What to notice:** "roughly balanced" matters because it means plain accuracy is a fair summary; a model cannot score well just by always guessing the most common activity. (The dataset calls lying down "laying"; we keep its label.)

Here is what the total acceleration looks like for one window of each activity:

In [23]:
# --- Plot one window of each activity (total acceleration only, so gravity is visible) ---
fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharey=True)   # 6 panels, one per activity, same y scale
t = np.arange(128) / 50                                       # 50 readings per second
for k, ax in enumerate(axes.flat):
    w = int(np.where(y_all == k)[0][0])                      # the first window of each activity
    # Channels 6, 7, 8 are total acceleration x, y, z (see SIGNALS above).
    for c, name in zip(range(6, 9), ("x", "y", "z")):
        ax.plot(t, X_all[w, :, c], label=f"acceleration {name}")
    ax.set_title(ACTIVITIES[k]); ax.set_xlabel("seconds"); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=7); axes[0, 0].set_ylabel("total acceleration (g)"); axes[1, 0].set_ylabel("total acceleration (g)")
plt.suptitle("2.56 seconds of phone sensor data, one window per activity"); plt.tight_layout(); plt.show()

/var/folders/wq/5ppc_s8x1x96s5xs9d2drg2r0000gp/T/ipykernel_33164/1985950799.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.suptitle("2.56 seconds of phone sensor data, one window per activity"); plt.tight_layout(); plt.show()


The three walking activities are rhythmic, with a peak at every step, though the differences between them are subtle. The three still activities are flat lines; what separates them is mostly *which way gravity points*. Lying down puts gravity on a different axis from sitting or standing. Sitting and standing look almost identical to a phone at the waist, so expect those two to be confused.

## 2. Splitting by Person

The test set already contains only new people. For **validation**, the data we use to choose models, we do the same: we hold out 4 whole people from the training group, rather than random windows. If windows from the same person appeared in both training and validation, validation would reward memorising each person's way of walking, and overestimate accuracy on new people.

We also **scale** each of the 9 channels to mean 0 and standard deviation 1, using statistics from the training people only.

Why scale at all? The channels come in different units and ranges (acceleration in g, rotation in radians per second), and neural networks train more smoothly when all inputs are on a similar scale. Why training people only? The scaling statistics are part of the model; computing them on validation or test people would let information about those people **leak** into training. The app later reuses exactly these statistics.

In [24]:
# --- Validation split by person, and channel scaling ---
# Take the last 4 people (by id) of the training group as validation. Whole people, not random windows:
# windows from one person look alike, so mixing them across train and validation would let the model
# "recognise the person" and make validation look better than it will be on new people.
val_people = [int(v) for v in sorted(set(s_all))[-4:]]         # hold out 4 whole people for validation
is_val = np.isin(s_all, val_people)                             # True for each validation person's window
X_tr, y_tr, X_val, y_val = X_all[~is_val], y_all[~is_val], X_all[is_val], y_all[is_val]
# Mean and standard deviation of each of the 9 channels, over every reading of every training window
# (reshape (N, 128, 9) -> (N*128, 9)). They come from training people only: using validation or test data
# here would leak information about the people we are supposed to be measuring on.
mu, sd = X_tr.reshape(-1, 9).mean(0), X_tr.reshape(-1, 9).std(0)   # per-channel scaling, from training people only
# Scale any array (..., 9) to mean 0, std 1 per channel. Same mu and sd for train, validation, test and the app.
norm = lambda X: (X - mu) / sd
print(f"train {len(X_tr)} windows ({len(set(s_all[~is_val]))} people) | validation {len(X_val)} windows (people {val_people}) | test {len(X_test)} windows")

train 5867 windows (17 people) | validation 1485 windows (people [27, 28, 29, 30]) | test 2947 windows


That leaves 5867 training windows from 17 people, and 1485 validation windows from people 27, 28, 29 and 30.

So we now have three groups of people that never overlap: 17 for **training** (the model learns from them), 4 for **validation** (we use them to choose between models and epochs) and 9 for **test** (looked at only at the end, to estimate how the chosen model does on new people).

> **Try it yourself (optional): split by window instead of by person.**
> In a new cell, shuffle all training windows and hold out a random 20% as validation (ignoring `s_all`), then retrain one model (e.g. `train("GRU")` after rebuilding `Xt`, `Xv`, `yt`, `yv`). **What to look for:** whether validation accuracy looks better than with the person split while test accuracy on the 9 new people does not improve. Any such gap is the over-optimism a person split protects you from. (Retraining overwrites `artifacts/` only if you also run the training cell; a separate call to `train` does not save anything.)

## 3. A Baseline Without Sequences

Before any recurrent network, a baseline: summarise each window by 4 numbers per channel (mean, standard deviation, minimum and maximum, 36 numbers in all), throw away the order of the readings, and fit logistic regression.

A **baseline** is a simple model that sets the bar: a deep network is only worth its extra cost if it beats it. **Logistic regression** is a linear classifier: it weights the 36 numbers, adds them up into a score per activity, and picks the highest score.

In [25]:
# --- Baseline: order-free summary statistics + logistic regression ---
def summary(X):                                              # 4 numbers per channel: mean, std, min, max
    """Summarise each window by 4 statistics per channel, ignoring the order of the readings.

    Input X: (N, 128, 9). Each statistic is taken over the time axis (axis 1), giving (N, 9).
    Output: (N, 36) - mean, std, min and max of all 9 channels side by side.
    Shuffling the 128 readings would not change these numbers, which is exactly the point of the baseline.
    """
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1)], axis=1)
# max_iter is raised so the solver has enough steps to converge.
base = LogisticRegression(max_iter=5000).fit(summary(norm(X_tr)), y_tr)
base_test = base.predict(summary(norm(X_test)))             # kept for the comparison table in Section 5
print(f"summary statistics + logistic regression: validation {base.score(summary(norm(X_val)), y_val):.3f} | test {(base_test == y_test).mean():.3f}")

summary statistics + logistic regression: validation 0.875 | test 0.819


The baseline reaches 0.875 on validation and 0.819 on the test people. Simple statistics already separate a lot, but they ignore *how* the signal changes over time. Its weakest activity, as Section 5 shows, is walking upstairs, where the rhythm matters.

**What to notice:** test accuracy is lower than validation accuracy. That is normal: the 9 test people are a different, larger group, and some of them move quite differently from the training people (Section 6).

## 4. Three Recurrent Networks

A **recurrent neural network** processes a sequence one time step at a time, using the same weights at every step. All three read the 128 readings one at a time, updating a **hidden state** of 64 numbers at each step, a running summary of what they've seen. After the last reading, a linear layer turns the final hidden state into scores for the six activities (Note 5.4).
- **RNN:** the plain version. At each step, the new state is a squashed mix of the previous state and the current reading. Over long sequences, the learning signal from early steps tends to fade (vanishing gradients).
- **LSTM** (long short-term memory): adds a separate memory cell and three **gates**, learned switches that decide what to forget, what to store and what to output. That lets information and gradients survive across many steps.
- **GRU** (gated recurrent unit): a simplified gated design with two gates and no separate cell. It is often as good as an LSTM, with fewer parameters.

**Vanishing gradients**, in a sentence: during training, the error signal is passed backwards through every time step, and in a plain RNN it is multiplied by similar factors at each step, so after many steps it can shrink towards zero; the early readings then have almost no influence on learning. Gates give the signal a more direct path back through time.

The cell below defines one model class that works for all three kinds, so the *only* difference between them is the recurrent layer. It then counts each model's **parameters** (the learnable weights).

In [26]:
# --- One model class for all three recurrent networks ---
class SequenceClassifier(nn.Module):
    """A recurrent layer reads the 128 readings in order; its final hidden state feeds a linear classifier.

    kind:      "RNN", "LSTM" or "GRU" - the only thing that differs between the three models, so the comparison is fair.
    hidden:    size of the hidden state (the network's running memory), 64 numbers.
    n_in:      inputs per time step, 9 sensor channels.
    n_classes: number of activities, 6.
    Input (batch, 128, 9) -> output (batch, 6) scores (logits), one per activity.
    """
    def __init__(self, kind, hidden=64, n_in=9, n_classes=6):
        """Build the recurrent layer of the chosen kind and the linear classification head."""
        super().__init__()
        rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]   # pick the layer class by name
        # batch_first=True means tensors are (batch, time, channels), matching our (N, 128, 9) data.
        self.rnn = rnn(n_in, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_classes)            # 64 hidden numbers -> 6 activity scores
    def forward(self, x):                                    # x: (B, 128, 9)
        """Read a batch of windows step by step and score each window.

        x: (B, 128, 9) scaled readings.
        Returns (B, 6) logits: unnormalised scores; softmax turns them into probabilities.
        """
        # The recurrent layer returns the hidden state after every time step, plus the final state(s),
        # which we ignore with "_" (for an LSTM that would also include the memory cell).
        out, _ = self.rnn(x)                                 # out: (B, 128, hidden), one state per time step
        # We classify from the LAST time step only: by then the hidden state has seen all 128 readings,
        # so it is a summary of the whole window.
        return self.head(out[:, -1])                         # (B, 6) from the last time step

# Count the learnable numbers (parameters) in each model. The recurrent layer's size depends on its number of
# gates: RNN has 1 set of weights, GRU 3, LSTM 4. The linear head is the same for all three.
for kind in ("RNN", "LSTM", "GRU"):
    m = SequenceClassifier(kind)
    print(f"{kind:<5} parameters: {sum(p.numel() for p in m.rnn.parameters()):>6,} in the recurrent layer, {sum(p.numel() for p in m.parameters()):>6,} in total")

RNN   parameters:  4,800 in the recurrent layer,  5,190 in total
LSTM  parameters: 19,200 in the recurrent layer, 19,590 in total
GRU   parameters: 14,400 in the recurrent layer, 14,790 in total


The recurrent layer has 4,800 parameters in the RNN, 19,200 in the LSTM (four sets of weights: three gates plus the candidate memory) and 14,400 in the GRU (three sets). The parameter counts follow directly from the number of gates.

**How to read this output:** "in total" adds the linear head, which is identical for all three models. The differences come entirely from the recurrent layer: the LSTM has four times the RNN's recurrent weights and the GRU three times.

Training uses the same loop for all three: Adam, batches of 64 windows, 12 epochs, gradient clipping (which recurrent networks often need), and the epoch with the best validation accuracy is kept.

A few terms: an **epoch** is one full pass over the training windows. **Adam** is a widely used optimiser, the rule that updates the weights from the gradients. The loss is **cross-entropy**, the standard loss for choosing one class out of several. **Gradient clipping** caps the overall size of the gradients before each update: backpropagating through 128 steps can occasionally produce a huge gradient ("exploding gradients") that would wreck the weights in one step, and clipping prevents that. Keeping the epoch with the best validation accuracy is a simple form of **early stopping**.

The next cell prepares the data as PyTorch tensors and defines two helpers: `predict` (labels for many windows) and `train` (the training loop). It only defines them; nothing is trained yet.

In [27]:
# --- Tensors, a prediction helper and the training loop (shared by all three models) ---
# Scale with the training statistics, then convert to PyTorch tensors: t = train, v = validation, s = test.
Xt, Xv, Xs = (torch.tensor(norm(X)) for X in (X_tr, X_val, X_test))   # each (N, 128, 9), float32
yt, yv, ys = (torch.tensor(y) for y in (y_tr, y_val, y_test))         # each (N,), labels 0-5

def predict(model, X):
    """Predict an activity label for every window in X.

    model: a SequenceClassifier. X: tensor (N, 128, 9), already scaled.
    Returns a tensor (N,) of predicted labels 0-5, on the CPU.
    Works in chunks of 512 windows to keep memory use small.
    """
    model.eval()                                             # evaluation mode (no training-only behaviour)
    with torch.no_grad():                                    # no gradients needed for prediction: faster, less memory
        # argmax(1) picks the activity with the highest score for each window.
        return torch.cat([model(X[i:i + 512].to(device)).argmax(1).cpu() for i in range(0, len(X), 512)])

def train(kind, epochs=12, seed=0):
    """Train one model of the given kind and keep the epoch with the best validation accuracy.

    kind: "RNN", "LSTM" or "GRU". epochs: passes over the training data. seed: fixes the random start
    (initial weights and batch order), so a run can be repeated exactly or varied on purpose.
    Returns (model with the best weights, list of validation accuracy per epoch, best epoch number, seconds taken).
    """
    torch.manual_seed(seed)                                  # same seed -> same initial weights
    model = SequenceClassifier(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)      # Adam: a standard, robust optimiser
    g = torch.Generator().manual_seed(seed)                  # separate generator so the batch order is reproducible too
    # best = (best validation accuracy so far, a copy of the weights at that point, the epoch it happened)
    best, history, t0 = (-1, None, 0), [], time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()                                        # training mode
        # Shuffle the training windows, then cut them into mini-batches of 64 indices.
        for b in torch.randperm(len(Xt), generator=g).split(64):
            opt.zero_grad()                                  # clear gradients left over from the previous batch
            # Forward pass, cross-entropy loss (the standard loss for classification), then backpropagation.
            F.cross_entropy(model(Xt[b].to(device)), yt[b].to(device)).backward()
            # Gradient clipping: if the gradients' overall size (norm) exceeds 1.0, scale them down to 1.0.
            # Backpropagating through 128 time steps can occasionally produce huge gradients ("exploding"
            # gradients) that wreck the weights in a single step; clipping prevents that.
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # keeps recurrent training stable
            opt.step()                                       # update the weights
        # After each epoch, measure accuracy on the 4 validation people (never on test).
        acc = (predict(model, Xv) == yv).float().mean().item()
        history.append(acc)
        if acc > best[0]:
            # .clone() makes a real copy; otherwise later training would overwrite the saved "best" weights.
            best = (acc, {k: v.clone() for k, v in model.state_dict().items()}, epoch)
    # Return the model as it was at its best validation epoch (a simple form of early stopping).
    model.load_state_dict(best[1])
    return model, history, best[2], time.perf_counter() - t0

Now the models themselves: on the first run each one is trained and saved; on later runs the saved weights are loaded. Either way, the cell finishes by measuring validation accuracy and choosing the best model.

In [28]:
# --- Train (first run) or load (later runs) the three models, then choose one on validation ---
os.makedirs("artifacts", exist_ok=True)                      # folder for saved models and app files
models, histories, info = {}, {}, {}
for kind in ("RNN", "LSTM", "GRU"):
    if TRAIN:
        models[kind], histories[kind], ep, secs = train(kind)
        torch.save(models[kind].state_dict(), f"artifacts/{kind.lower()}.pt")   # reused by later runs and the app
        info[kind] = {"best_epoch": ep, "train_time": f"{secs:.0f} s"}
        print(f"{kind:<5} trained in {secs:.0f} s | best epoch {ep} | validation accuracy {max(histories[kind]):.3f}")
    else:
        # Rebuild an empty model of the same shape, then fill it with the saved weights.
        models[kind] = SequenceClassifier(kind).to(device)
        models[kind].load_state_dict(torch.load(f"artifacts/{kind.lower()}.pt")); print(f"loaded artifacts/{kind.lower()}.pt")
val_acc = {kind: (predict(models[kind], Xv) == yv).float().mean().item() for kind in models}
# Pick the model with the highest validation accuracy. We never choose using the test set: test must stay an
# honest, untouched estimate of performance on new people.
best_kind = max(val_acc, key=val_acc.get)                          # chosen on validation, never on test
print("validation accuracy:", {k: round(v, 3) for k, v in val_acc.items()}, "| chosen for the app:", best_kind)
# Save the scaling statistics and activity names: the app must scale new data exactly as in training.
json.dump({"mu": mu.tolist(), "sd": sd.tolist(), "activities": ACTIVITIES}, open("artifacts/scaling.json", "w"))

loaded artifacts/rnn.pt
loaded artifacts/lstm.pt
loaded artifacts/gru.pt
validation accuracy: {'RNN': 0.883, 'LSTM': 0.91, 'GRU': 0.937} | chosen for the app: GRU


LSTM  trained in 20 s | best epoch 11 | validation accuracy 0.910


GRU   trained in 16 s | best epoch 10 | validation accuracy 0.937
validation accuracy: {'RNN': 0.883, 'LSTM': 0.91, 'GRU': 0.937} | chosen for the app: GRU


All three train in seconds on a CPU, the RNN fastest because it has the least work per step. On the 4 validation people, the RNN reaches 0.883, the LSTM 0.910 and the GRU 0.937, so we choose the **GRU** for the app. That choice is made on validation, never on test. The learning curves show the difference in training behaviour:

*(The learning-curve plot below appears only on a run that trains the models, that is when `TRAIN` is `True`. On a run that loads saved models, the cell shows nothing; the description that follows refers to the training run.)*

In [29]:
# --- Learning curves: validation accuracy after each epoch ---
# Histories only exist when the models were trained in this session, so on a load-only run nothing is drawn.
if histories:
    plt.figure(figsize=(6.5, 3.6))
    for kind, h in histories.items():
        plt.plot(range(1, len(h) + 1), h, "o-", label=kind)   # one line per model, one dot per epoch
    plt.xlabel("epoch"); plt.ylabel("validation accuracy (4 unseen people)"); plt.title("Learning curves")
    plt.legend(); plt.grid(alpha=0.3); plt.show()

The gated networks learn quickly and smoothly, reaching high validation accuracy within a few epochs. The plain RNN starts lower, wobbles from epoch to epoch, and is still improving at the end. That's the practical face of the vanishing-gradient problem: with 128 steps to look back through, the plain RNN learns more slowly and less reliably.

> **Try it yourself (optional): change the hidden size.**
> In `SequenceClassifier`, change `hidden=64` to `16`, then to `128`, and re-run from the model cell with `TRAIN = True`. **What to look for:** how the parameter counts change (they grow much faster than the hidden size), how training time changes, and whether validation accuracy actually improves or just costs more.

> **Try it yourself (optional): remove gradient clipping.**
> Comment out the `nn.utils.clip_grad_norm_(...)` line in `train`, then retrain with `TRAIN = True`. **What to look for:** whether the learning curves, especially the plain RNN's, become more jagged, or whether accuracy suddenly drops in some epoch. The effect may be small on some runs; try a couple of seeds.

## 5. Comparing on New People

Now the test set, 9 people none of the models has seen. Here is the accuracy for each activity (the share of that activity's windows recognised correctly):

In [30]:
# --- Test accuracy overall and per activity, for the baseline and the three networks ---
preds = {"summary statistics + logistic regression": torch.tensor(base_test)}   # baseline test predictions
preds.update({kind: predict(models[kind], Xs) for kind in ("RNN", "LSTM", "GRU")})
# Header row: model name, overall test accuracy, then the first 10 letters of each activity name.
print(f"{'model':<42} {'test acc':>8} " + " ".join(f"{a[:10]:>10}" for a in ACTIVITIES))
for name, p in preds.items():
    # Per-activity accuracy (recall): of the windows that truly are activity k, what share was predicted as k?
    rec = [((p == k) & (ys == k)).sum().item() / (ys == k).sum().item() for k in range(6)]
    print(f"{name:<42} {(p == ys).float().mean():>8.3f} " + " ".join(f"{r:>10.2f}" for r in rec))

model                                      test acc    walking walking up walking do    sitting   standing     laying
summary statistics + logistic regression      0.819       0.87       0.61       0.78       0.79       0.89       0.95
RNN                                           0.853       0.80       0.83       0.88       0.78       0.86       0.95
LSTM                                          0.903       0.95       0.89       0.99       0.82       0.84       0.95
GRU                                           0.898       0.98       0.93       0.94       0.78       0.82       0.95


**How to read this table:** each row is a model. "test acc" is the share of all test windows labelled correctly. Each following column is one activity (names cut to 10 letters) and shows the share of that activity's windows recognised correctly, also called **recall**.

All three recurrent networks beat the statistics baseline (0.819). The plain RNN reaches 0.853, and the two gated networks are close together: the LSTM 0.903 and the GRU 0.898. The biggest gains over the baseline are on the walking activities, where order matters: walking upstairs rises from 0.61 to 0.93 with the GRU. The still activities barely improve: sitting stays around 0.78–0.82 for every model.

The confusion matrix for the GRU shows exactly where it goes wrong. A **confusion matrix** counts, for every true activity (rows), how often each activity was predicted (columns); the diagonal holds the correct answers and every other square is a particular kind of mistake.

In [31]:
# --- Confusion matrix of the chosen model on the test people ---
# cm[true, predicted] counts windows. The diagonal holds correct answers; everything else is a mistake.
cm = np.zeros((6, 6), dtype=int)
for a, b in zip(y_test, preds[best_kind].numpy()):
    cm[a, b] += 1
fig, ax = plt.subplots(figsize=(5.5, 4.6))
ax.imshow(cm, cmap="Blues")                                   # darker blue = more windows
# Write each count in its square; white text on dark squares so it stays readable.
for i in range(6):
    for j in range(6):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xticks(range(6)); ax.set_xticklabels(ACTIVITIES, rotation=45, ha="right"); ax.set_yticks(range(6)); ax.set_yticklabels(ACTIVITIES)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{best_kind} on test (9 new people)"); plt.tight_layout(); plt.show()

/var/folders/wq/5ppc_s8x1x96s5xs9d2drg2r0000gp/T/ipykernel_33164/2812275514.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{best_kind} on test (9 new people)"); plt.tight_layout(); plt.show()


The largest errors are **sitting vs standing**: 99 sitting windows called standing and 92 standing windows called sitting. A phone at the waist sees almost the same thing in both postures, so this is a limit of the *sensor*, not of the network. A different sensor position, or extra information (for example, how the person got there), would be needed to separate them.

### Is the LSTM really better than the GRU on test?

The LSTM scored 0.903 on test and the GRU 0.898, even though the GRU won on validation. Is either really better? We retrain each model with three more random seeds:

Each seed gives the network a different random starting point and a different order of batches, so it ends up slightly different. If the gap between two models is smaller than the spread between seeds of the *same* model, the gap is probably luck.

In [32]:
# --- Seed study: retrain each model with 3 different seeds to see how much results vary by chance ---
# A single run depends on its random start. If two models' gap is smaller than the run-to-run spread,
# we cannot honestly say one is better. (Only on a training run; otherwise we load the saved results.)
if TRAIN:                                                     # three seeds per model: is the ranking real?
    study = {}
    for kind in ("RNN", "LSTM", "GRU"):
        runs = []
        for seed in (1, 2, 3):                                # seed 0 was the main run above
            m, h, _, _ = train(kind, seed=seed)
            # Store (validation accuracy, test accuracy) for this run.
            runs.append(((predict(m, Xv) == yv).float().mean().item(), (predict(m, Xs) == ys).float().mean().item()))
        study[kind] = runs
    json.dump(study, open("artifacts/seed_study.json", "w"))   # saved so later runs skip the 9 extra trainings
else:
    study = json.load(open("artifacts/seed_study.json"))
# Report the lowest-highest range across the 3 seeds, for validation and for test.
for kind, runs in study.items():
    v, t_ = [r[0] for r in runs], [r[1] for r in runs]
    print(f"{kind:<5} 3 more seeds: validation {min(v):.3f}-{max(v):.3f} | test {min(t_):.3f}-{max(t_):.3f}")

RNN   3 more seeds: validation 0.872-0.890 | test 0.857-0.867
LSTM  3 more seeds: validation 0.914-0.924 | test 0.887-0.905
GRU   3 more seeds: validation 0.918-0.928 | test 0.871-0.890


**How to read this output:** for each model, the lowest and highest accuracy across the three extra seeds, on validation and on test. Overlapping ranges mean the models cannot be told apart reliably from these runs.

Across seeds, the plain RNN stays clearly behind (test 0.857–0.867). The LSTM (test 0.887–0.905) and the GRU (test 0.871–0.890) overlap on validation and come close on test. The gap between them is about the size of the seed-to-seed variation, so with these runs we can't call either clearly better. The robust conclusion: **gated networks beat the plain RNN here**, and the GRU gets similar results to the LSTM with fewer parameters. Single runs can mislead; repeating them is cheap insurance.

> **Try it yourself (optional): does the order of the readings really matter?**
> No retraining needed. In a new cell, shuffle the time axis of the test windows and measure the GRU again, for example: `perm = torch.randperm(128)` then `(predict(models["GRU"], Xs[:, perm]) == ys).float().mean()`. Also try reversing time with `Xs.flip(1)`. **What to look for:** how far accuracy falls compared with the table above, and which activities suffer most (walking ones or still ones?). The baseline's summary statistics would not change at all under shuffling.

## 6. Does It Work for Everyone?

Average accuracy hides how the model does for individual people. Here is the GRU's accuracy for each of the 9 test people:

In [33]:
# --- Accuracy for each test person separately ---
p = preds[best_kind].numpy()                                  # the chosen model's test predictions
# For each person id, the share of that person's windows predicted correctly.
per_person = {int(s): float((p[s_test == s] == y_test[s_test == s]).mean()) for s in sorted(set(s_test))}
print(f"{best_kind} accuracy for each of the 9 test people:", {s: round(a, 3) for s, a in per_person.items()})
print(f"range: {min(per_person.values()):.3f} to {max(per_person.values()):.3f}")

GRU accuracy for each of the 9 test people: {2: 0.887, 4: 0.886, 9: 0.764, 10: 0.592, 12: 0.991, 13: 0.96, 18: 0.959, 20: 0.963, 24: 1.0}
range: 0.592 to 1.000


**How to read this output:** the keys are the test people's ids from the original dataset, and each value is the share of that person's windows the GRU labelled correctly.

Accuracy ranges from 0.592 to 1.000. For most people the model is above 0.88, but for person 10 it gets only 0.592, and for person 9, 0.764. People move differently, and a model trained on 17 people may simply not have seen someone who walks, sits or stands like person 10. In a real product, this is why teams collect data from a wide range of users, report per-user results rather than only the average, and often let the app adapt to each user over time.

> **Try it yourself (optional): where does person 10 go wrong?**
> Build the confusion matrix from Section 5 using only person 10's windows (select them with `s_test == 10`). **What to look for:** whether the errors are the same sitting/standing confusion as for everyone else, or a different kind of mistake.

## 7. The Application

The app takes a 2.56-second window of the 9 sensor channels and returns the most likely activities. Here it is on the first test window of each activity (all from test person 2):

The cell below saves these example windows for the app, defines `recognise` (raw window in, top-3 activities with probabilities out) and records which model the app should load. The probabilities come from **softmax**, which turns the model's 6 raw scores into numbers between 0 and 1 that add up to 1.

In [34]:
# --- Example recordings for the app, and the prediction function it uses ---
examples = {}
for k, a in enumerate(ACTIVITIES):
    w = int(np.where(y_test == k)[0][0])                     # the first test window of each activity
    examples[f"{a} (test person {s_test[w]})"] = X_test[w]   # raw (unscaled) (128, 9) window, named for the dropdown
np.savez("artifacts/examples.npz", **{n: v for n, v in examples.items()})   # the app loads these as its examples
model = models[best_kind]                                    # the model chosen on validation (the GRU here)
def recognise(window):
    """128 x 9 raw readings in, activity probabilities out.

    window: numpy array (128, 9) of raw, unscaled readings.
    Returns a dict of the top 3 activities and their probabilities (rounded to 3 decimals), most likely first.
    """
    # window[None] adds a batch axis: (128, 9) -> (1, 128, 9). Then scale with the TRAINING statistics.
    x = torch.tensor(norm(window[None].astype(np.float32))).to(device)
    model.eval()
    with torch.no_grad():
        # softmax turns the 6 scores into probabilities that sum to 1; [0] takes the single window in the batch.
        prob = model(x).softmax(1)[0].cpu().numpy()
    # Pair each activity with its probability, sort from most to least likely, keep the top 3.
    return {a: round(float(q), 3) for a, q in sorted(zip(ACTIVITIES, prob), key=lambda t: -t[1])[:3]}
for name, w in examples.items():
    print(f"{name:<36} -> {recognise(w)}")
json.dump({"model": best_kind}, open("artifacts/app_model.json", "w"))   # tells app.py which model file to load

walking (test person 2)              -> {'walking': 0.999, 'walking upstairs': 0.0, 'walking downstairs': 0.0}
walking upstairs (test person 2)     -> {'walking upstairs': 0.999, 'walking': 0.0, 'standing': 0.0}
walking downstairs (test person 2)   -> {'walking downstairs': 0.999, 'walking': 0.001, 'walking upstairs': 0.0}
sitting (test person 2)              -> {'standing': 0.834, 'sitting': 0.164, 'walking': 0.001}
standing (test person 2)             -> {'standing': 0.994, 'sitting': 0.004, 'walking upstairs': 0.001}
laying (test person 2)               -> {'laying': 1.0, 'sitting': 0.0, 'walking downstairs': 0.0}


Walking, the stairs, standing and lying down are recognised confidently. The sitting window is called standing, with 0.834 probability, exactly the confusion the matrix predicted, and a good thing to show students: confident is not the same as correct.

`app.py` in this folder is a small **Gradio** app. Pick one of these recordings from people the model never saw, or upload your own as a CSV file (128 rows × 9 columns, recorded at 50 readings per second), and it plots the readings and shows the top three activities.

The app does not retrain anything: it reads the files this notebook saved in `artifacts/` (the chosen model's weights, the scaling statistics and the example windows). A CSV you upload must use the same 9 channels in the same order as the notebook: body acceleration x, y, z; body rotation x, y, z; total acceleration x, y, z. To open the app inside the notebook, set `LAUNCH_APP = True` below; to run it on its own, run `python app.py` in this folder.

In [35]:
# --- Optionally open the Gradio app inside the notebook ---
LAUNCH_APP = True                                      # set True in class to open the app below this cell
if LAUNCH_APP:
    import app                                         # app.py in this folder; it loads the files saved in artifacts/
    app.demo.launch()                                  # starts a local web server and shows the app here

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


## What We Learned

- **Sensor data is a sequence,** and order matters. Summary statistics (logistic regression) reached 0.819 on new people; recurrent networks that read the sequence did better.
- **Split by person.** Validation and test used only people the model hadn't seen, which is the question a real app has to answer.
- **Gates help.** The plain RNN reached 0.853 on test and learned slowly and unsteadily; the LSTM (0.903) and GRU (0.898) learned fast and well.
- **LSTM vs GRU:** across seeds their difference was about the size of the run-to-run variation. The GRU used fewer parameters (14,400 vs 19,200 in the recurrent layer) for similar accuracy.
- **Know the limits of your sensor.** Sitting and standing were confused in both directions, because a waist-worn phone barely tells them apart.
- **Check individuals, not just the average.** Accuracy per test person ranged from 0.592 to 1.000.
- **Fit preprocessing on training data only, and reuse it everywhere.** The scaling statistics came from the training people and were saved so the app scales new recordings in exactly the same way.
- **Choose on validation, report on test.** The GRU was picked on validation; the test set was only used to measure the final result.

| Model | Recurrent-layer parameters | Validation (chosen on) | Test (9 new people) | Test, 3 more seeds |
|---|---|---|---|---|
| Summary statistics + logistic regression | — | 0.875 | 0.819 | — |
| RNN | 4,800 | 0.883 | 0.853 | 0.857–0.867 |
| LSTM | 19,200 | 0.910 | 0.903 | 0.887–0.905 |
| GRU (used in the app) | 14,400 | 0.937 | 0.898 | 0.871–0.890 |

### Key terms

- **Window:** one fixed-length slice of a recording (here 128 readings, 2.56 seconds), used as one example.
- **Channel:** one sensor signal recorded at every time step (here 9: body acceleration, rotation and total acceleration, each on x, y and z).
- **Recurrent neural network (RNN):** a network that reads a sequence one step at a time with the same weights, carrying a hidden state forward.
- **Hidden state:** the network's running summary of everything it has read so far; here 64 numbers, updated at every time step.
- **Vanishing gradients:** the learning signal shrinking as it is passed back through many time steps, so early inputs barely affect learning.
- **Exploding gradients:** the opposite problem: the signal growing so large that one update wrecks the weights.
- **Gradient clipping:** scaling the gradients down whenever their overall size exceeds a limit, to prevent exploding gradients.
- **Gate:** a learned switch (values between 0 and 1) inside an LSTM or GRU that controls what is kept, forgotten or passed on.
- **LSTM / GRU:** gated recurrent networks; the LSTM has three gates and a separate memory cell, the GRU two gates and no separate cell.
- **Parameters:** the learnable weights of a model.
- **Epoch:** one full pass through the training data.
- **Validation set:** data held out from training and used to choose models and settings; here 4 whole people.
- **Test set:** data used only once at the end to estimate performance on new people; here 9 people.
- **Data leakage:** information from validation or test data sneaking into training, which makes results look better than they really are.
- **Seed:** the starting value of a random number generator; fixing it makes a run repeatable, changing it shows how much results vary by chance.
- **Baseline:** a simple reference model that a more complex model must beat to be worth it.
- **Recall (per-activity accuracy):** of the windows that truly are a given activity, the share the model recognised correctly.
- **Confusion matrix:** a table counting true activity against predicted activity; off-diagonal squares are mistakes.
- **Softmax:** turns a model's raw scores into probabilities that add up to 1.